# 🎙️ TurboVoiceCloner (Final Stable Fix)
**Features:** Anti-Disconnect | Python 3.10 Fix | Auto-GitHub Push

In [ ]:
# @title 🚀 Step 1: Python 3.10 & Stable TTS Setup
import os
os.environ['MPLBACKEND'] = 'Agg'
print("Installing Stable Environment...")
!sudo apt-get install python3.10 python3.10-dev python3.10-distutils -y -q
!wget https://bootstrap.pypa.io/get-pip.py -q && python3.10 get-pip.py -q
!python3.10 -m pip install -q coqui-tts gradio==4.44.1 pydub numpy<2.0.0
print("✅ Setup Complete!")

In [ ]:
# @title ⚙️ Step 2: Turbo Launch (Silence Remover Fixed)
with open("app_final.py", "w") as f:
    f.write('''
import os
os.environ['MPLBACKEND'] = 'Agg'
import gradio as gr
from TTS.api import TTS
from pydub import AudioSegment, silence

device = "cuda" if os.path.exists("/dev/nvidia0") else "cpu"
tts = TTS("tts_models/multilingual/multi-dataset/your_tts").to(device)

def process(text, ref, clean):
    out, final = "temp.wav", "final_voice.wav"
    tts.tts_to_file(text=text, speaker_wav=ref, language="en", file_path=out)
    if clean:
        audio = AudioSegment.from_file(out)
        chunks = silence.split_on_silence(audio, min_silence_len=300, silence_thresh=-40, keep_silence=100)
        combined = AudioSegment.empty()
        for c in chunks: combined += c
        combined.export(final, format="wav")
    else:
        os.rename(out, final)
    return final

gr.Interface(fn=process, inputs=[gr.Textbox(label="Text"), gr.Audio(label="Voice Sample", type="filepath"), gr.Checkbox(label="Silence Remover", value=True)], outputs=gr.Audio(label="Output")).launch(share=True)
''')

!python3.10 app_final.py

In [ ]:
# @title 📂 Step 3: Auto-Push to GitHub
TOKEN = "YOUR_GITHUB_TOKEN"
USER = "shriramnag"
REPO = "TurboVoiceCloner"

!git config --global user.email "your-email@example.com"
!git config --global user.name "shriramnag"
!mkdir -p outputs
!cp final_voice.wav outputs/
!git add .
!git commit -m "New Voice Uploaded"
!git push https://{TOKEN}@github.com/{USER}/{REPO}.git main